# 3.1 — Model Training & Evaluation (Fixed)

**Perbaikan dari audit:**
- ✅ Naive baseline comparison (random walk: y_pred = y_{t-1})
- ✅ Train metrics ditampilkan (deteksi overfitting)
- ✅ Per-province RMSE breakdown
- ✅ Feature set yang sudah diperbaiki dari 2.2
- ✅ provinsi_id sebagai fitur (panel awareness)

**Input:** `2_data_preprocessing/output/2.2_final_feature_set.csv`

**Output:**
- `3_modelling/output/model_metrics.csv`
- `3_modelling/output/3_model_predictions.csv`
- `3_modelling/output/feature_importance.csv`
- `3_modelling/output/per_province_metrics.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    import xgboost as xgb
    use_xgboost = True
    model_name = 'XGBoost'
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    use_xgboost = False
    model_name = 'RandomForest'

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv').exists():
            return p
    raise FileNotFoundError('Could not find 2.2_final_feature_set.csv')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv'
output_dir = ROOT / '3_modelling' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} cols')

In [ ]:
# === Definisi kolom fitur ===
kolom_non_prediktor = ['nama_provinsi', 'tanggal', 'tahun', 'bulan', 'twp90_pct']
X_cols = [col for col in df.columns if col not in kolom_non_prediktor]

# Drop baris dengan missing pada lag features (baris awal dari tiap provinsi)
lag_cols = [c for c in X_cols if '_lag_' in c]
df_model = df.dropna(subset=lag_cols + ['twp90_pct']).copy()
df_model['tahun'] = df_model['tahun'].astype(int)

print(f'Features ({len(X_cols)}): {X_cols}')
print(f'Rows available for modelling: {len(df_model):,}')

# === Time-based split: Train <= 2024 | Test = 2025 ===
train_data = df_model[df_model['tahun'] <= 2024].copy()
test_data = df_model[df_model['tahun'] == 2025].copy()

X_train = train_data[X_cols]
y_train = train_data['twp90_pct']
X_test = test_data[X_cols]
y_test = test_data['twp90_pct']

print(f'Train: {X_train.shape} (years: {train_data["tahun"].min()}-{train_data["tahun"].max()})')
print(f'Test:  {X_test.shape} (year: {test_data["tahun"].unique()})')

In [ ]:
# === Train model ===
if use_xgboost:
    # Split training for early stopping
    train_sorted = train_data.sort_values('tanggal').reset_index(drop=True)
    val_size = int(0.2 * len(train_sorted))
    train_part = train_sorted.iloc[:-val_size]
    val_part = train_sorted.iloc[-val_size:]

    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric='rmse',
        early_stopping_rounds=50,
        random_state=42,
        enable_categorical=False,
    )
    model.fit(
        train_part[X_cols], train_part['twp90_pct'],
        eval_set=[(val_part[X_cols], val_part['twp90_pct'])],
        verbose=False,
    )
else:
    model = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

# === Predictions ===
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# === Naive baseline: random walk (y_pred = y_{t-1}) ===
if 'twp90_lag_1' in test_data.columns:
    y_naive = test_data['twp90_lag_1'].values
else:
    y_naive = y_test.shift(1).bfill().values

# === Metrics ===
def calc_metrics(y_true, y_pred, name):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2}

m_train = calc_metrics(y_train, y_train_pred, f'{model_name} (Train)')
m_test = calc_metrics(y_test, y_test_pred, f'{model_name} (Test)')
m_naive = calc_metrics(y_test, y_naive, 'Naive (Random Walk)')

print('='*60)
print(f'{"Model":<30} {"RMSE":>8} {"MAE":>8} {"R2":>8}')
print('-'*60)
for m in [m_train, m_test, m_naive]:
    print(f'{m["model"]:<30} {m["rmse"]:>8.4f} {m["mae"]:>8.4f} {m["r2"]:>8.4f}')
print('='*60)

# Overfitting check
gap = abs(m_train['rmse'] - m_test['rmse']) / m_test['rmse'] * 100
print(f'\nOverfitting gap (train-test RMSE): {gap:.1f}%')
if gap > 30:
    print('  ⚠️ WARNING: Large gap suggests overfitting')
else:
    print('  ✅ Gap within acceptable range')

# Model vs Naive
if m_test['rmse'] < m_naive['rmse']:
    print(f'\n✅ Model beats naive baseline by {((m_naive["rmse"] - m_test["rmse"]) / m_naive["rmse"] * 100):.1f}%')
else:
    print(f'\n🔴 Model WORSE than naive baseline!')

# Save metrics
metrics_df = pd.DataFrame([m_train, m_test, m_naive])
metrics_df.to_csv(output_dir / 'model_metrics.csv', index=False)
print(f'Saved: {output_dir / "model_metrics.csv"}')

In [ ]:
# === Per-Province RMSE Breakdown ===
pred_df = test_data[['provinsi_id', 'nama_provinsi', 'tanggal', 'tahun', 'bulan', 'twp90_pct']].copy()
pred_df = pred_df.rename(columns={'twp90_pct': 'y_true'})
pred_df['y_pred'] = y_test_pred
pred_df['y_naive'] = y_naive
pred_df['error'] = pred_df['y_pred'] - pred_df['y_true']

prov_metrics = []
for prov_id, grp in pred_df.groupby('provinsi_id'):
    prov_name = grp['nama_provinsi'].iloc[0]
    rmse_model = np.sqrt(mean_squared_error(grp['y_true'], grp['y_pred']))
    rmse_naive = np.sqrt(mean_squared_error(grp['y_true'], grp['y_naive']))
    prov_metrics.append({
        'provinsi_id': prov_id,
        'nama_provinsi': prov_name,
        'rmse_model': round(rmse_model, 5),
        'rmse_naive': round(rmse_naive, 5),
        'beats_naive': rmse_model < rmse_naive,
    })

prov_df = pd.DataFrame(prov_metrics).sort_values('rmse_model', ascending=False)
print(f'\n=== Per-Province Evaluation ===')
print(f'Provinces where model beats naive: {prov_df["beats_naive"].sum()}/{len(prov_df)}')
display(prov_df)

prov_df.to_csv(output_dir / 'per_province_metrics.csv', index=False)
pred_df.to_csv(output_dir / '3_model_predictions.csv', index=False)
print(f'Saved predictions and per-province metrics')

In [ ]:
# === Feature Importance (full, no dedup) ===
if hasattr(model, 'feature_importances_'):
    fi_df = pd.DataFrame({
        'feature': X_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    fi_df['rank'] = range(1, len(fi_df) + 1)
    fi_df['importance_pct'] = fi_df['importance'] / fi_df['importance'].sum()

    print('\n=== Top 15 Feature Importance ===')
    display(fi_df.head(15))

    fi_df.to_csv(output_dir / 'feature_importance.csv', index=False)
    print(f'Saved: {output_dir / "feature_importance.csv"}')
else:
    print('Model does not have feature_importances_ attribute')